In [1]:
from pathlib import Path
import sys

import duckdb


# ============================================================
# PROJECT CONFIG
# ============================================================

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in [current, *current.parents]
    if (p / "src" / "data_pipeline" / "config.py").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src" / "data_pipeline")
)

from config import (
    USER_DAY_FILE,
    USER_BASELINE_FILE,
    BEHAVIOR_FEATURES_FILE,
    EMAIL_CSV,
)


# ============================================================
# HELPERS
# ============================================================

def db_path(path):
    return Path(path).resolve().as_posix()


# ============================================================
# INPUT CHECK
# ============================================================

required_files = {
    "User-Day dataset": USER_DAY_FILE,
    "User baseline": USER_BASELINE_FILE,
}

missing = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not Path(path).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing)
    )

print("✓ Required input files found.")


# ============================================================
# PATHS
# ============================================================

user_day = db_path(USER_DAY_FILE)
baseline = db_path(USER_BASELINE_FILE)
output = db_path(BEHAVIOR_FEATURES_FILE)


# ============================================================
# BUILD BEHAVIOR FEATURES
# ============================================================

print("\n" + "=" * 70)
print("BUILDING USER-DAY BEHAVIOR FEATURES")
print("=" * 70)

con = duckdb.connect()

con.execute(f"""
    COPY (

        WITH joined AS (

            SELECT
                d.*,
                b.* EXCLUDE(user),

                CASE
                    WHEN d.first_logon IS NOT NULL
                    THEN
                        EXTRACT(HOUR FROM d.first_logon) * 60
                        +
                        EXTRACT(MINUTE FROM d.first_logon)
                END AS first_logon_minute,

                CASE
                    WHEN d.last_logoff IS NOT NULL
                    THEN
                        EXTRACT(HOUR FROM d.last_logoff) * 60
                        +
                        EXTRACT(MINUTE FROM d.last_logoff)
                END AS last_logoff_minute

            FROM read_parquet(
                '{user_day}'
            ) d

            LEFT JOIN read_parquet(
                '{baseline}'
            ) b
                USING(user)
        ),

        daily AS (

            SELECT
                *,

                -- ===========================================
                -- BASELINE RATIOS
                -- ===========================================

                email_count
                    / GREATEST(
                        email_count_p95,
                        1
                    )
                    AS email_p95_ratio,

                external_email_events
                    / GREATEST(
                        external_email_p95,
                        1
                    )
                    AS external_email_p95_ratio,

                usb_connect_count
                    / GREATEST(
                        usb_connect_p95,
                        1
                    )
                    AS usb_connect_p95_ratio,

                files_copied_to_usb 
                    / GREATEST(
                        usb_files_p95,
                        1
                    )
                    AS usb_files_p95_ratio, 

                http_request_count
                    / GREATEST(
                        http_requests_p95,
                        1
                    )
                    AS http_p95_ratio,

                unique_domains
                    / GREATEST(
                        unique_domains_p95,
                        1
                    )
                    AS domains_p95_ratio,


                -- ===========================================
                -- ABOVE-P95 FLAGS
                -- ===========================================

                CASE
                    WHEN email_count
                         > email_count_p95
                    THEN 1
                    ELSE 0
                END AS email_above_p95,

                CASE
                    WHEN external_email_events
                         > external_email_p95
                    THEN 1
                    ELSE 0
                END AS external_email_above_p95,

                CASE
                    WHEN usb_connect_count
                         > usb_connect_p95
                    THEN 1
                    ELSE 0
                END AS usb_connect_above_p95,

                CASE
                    WHEN files_copied_to_usb
                         > usb_files_p95
                    THEN 1
                    ELSE 0
                END AS usb_files_above_p95,

                CASE
                    WHEN http_request_count
                         > http_requests_p95
                    THEN 1
                    ELSE 0
                END AS http_above_p95,

                CASE
                    WHEN unique_domains
                         > unique_domains_p95
                    THEN 1
                    ELSE 0
                END AS domains_above_p95,


                -- ===========================================
                -- UNUSUAL WORKING TIME
                -- ===========================================

                CASE
                    WHEN first_logon_minute IS NOT NULL
                     AND first_logon_minute
                         < first_logon_p05
                    THEN
                        first_logon_p05
                        - first_logon_minute
                    ELSE 0
                END AS early_logon_minutes,

                CASE
                    WHEN last_logoff_minute IS NOT NULL
                     AND last_logoff_minute
                         > last_logoff_p95
                    THEN
                        last_logoff_minute
                        - last_logoff_p95
                    ELSE 0
                END AS late_logoff_minutes,


                -- ===========================================
                -- DAILY CHANNEL ACTIVITY
                -- ===========================================

                CASE
                    WHEN email_count > 0
                    THEN 1
                    ELSE 0
                END AS email_active,

                CASE
                    WHEN usb_connect_count > 0
                      OR files_copied_to_usb > 0
                    THEN 1
                    ELSE 0
                END AS usb_active,

                CASE
                    WHEN http_request_count > 0
                    THEN 1
                    ELSE 0
                END AS http_active

            FROM joined
        ),

        with_flags AS (

            SELECT
                *,

                (
                    email_above_p95
                    + external_email_above_p95
                    + usb_connect_above_p95
                    + usb_files_above_p95
                    + http_above_p95
                    + domains_above_p95
                ) AS daily_abnormal_feature_count,

                CASE
                    WHEN (
                        email_above_p95
                        + external_email_above_p95
                        + usb_connect_above_p95
                        + usb_files_above_p95
                        + http_above_p95
                        + domains_above_p95
                    ) > 0
                    THEN 1
                    ELSE 0
                END AS abnormal_day

            FROM daily
        )

        SELECT
            *,

            -- ===============================================
            -- 7-DAY ROLLING FEATURES
            -- ===============================================

            SUM(
                external_email_events
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    6 PRECEDING
                    AND CURRENT ROW
            ) AS external_emails_7d,

            SUM(
                files_copied_to_usb
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    6 PRECEDING
                    AND CURRENT ROW
            ) AS usb_files_7d,

            SUM(
                http_request_count
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    6 PRECEDING
                    AND CURRENT ROW
            ) AS http_requests_7d,

            SUM(
                abnormal_day
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    6 PRECEDING
                    AND CURRENT ROW
            ) AS abnormal_days_7d,


            -- ===============================================
            -- 14-DAY ROLLING FEATURES
            -- ===============================================

            SUM(
                external_email_events
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    13 PRECEDING
                    AND CURRENT ROW
            ) AS external_emails_14d,

            SUM(
                files_copied_to_usb
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    13 PRECEDING
                    AND CURRENT ROW
            ) AS usb_files_14d,

            SUM(
                http_request_count
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    13 PRECEDING
                    AND CURRENT ROW
            ) AS http_requests_14d,

            SUM(
                abnormal_day
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    13 PRECEDING
                    AND CURRENT ROW
            ) AS abnormal_days_14d,


            -- ===============================================
            -- 30-DAY ROLLING FEATURES
            -- ===============================================

            SUM(
                files_copied_to_usb
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    29 PRECEDING
                    AND CURRENT ROW
            ) AS usb_files_30d,

            SUM(
                external_email_events
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    29 PRECEDING
                    AND CURRENT ROW
            ) AS external_emails_30d,

            SUM(
                abnormal_day
            ) OVER (
                PARTITION BY user
                ORDER BY day
                ROWS BETWEEN
                    29 PRECEDING
                    AND CURRENT ROW
            ) AS abnormal_days_30d,


            -- ===============================================
            -- CHANNEL DIVERSITY
            -- ===============================================
            -- Number of Email / USB / HTTP channels that
            -- were active at least once during the last
            -- seven User-Day records.
            --
            -- NOTE:
            -- This measures channel diversity, NOT rotation.
            -- ===============================================

            (
                MAX(
                    email_active
                ) OVER (
                    PARTITION BY user
                    ORDER BY day
                    ROWS BETWEEN
                        6 PRECEDING
                        AND CURRENT ROW
                )

                +

                MAX(
                    usb_active
                ) OVER (
                    PARTITION BY user
                    ORDER BY day
                    ROWS BETWEEN
                        6 PRECEDING
                        AND CURRENT ROW
                )

                +

                MAX(
                    http_active
                ) OVER (
                    PARTITION BY user
                    ORDER BY day
                    ROWS BETWEEN
                        6 PRECEDING
                        AND CURRENT ROW
                )

            ) AS activity_channel_diversity_7d

        FROM with_flags

        ORDER BY
            user,
            day

    )
    TO '{output}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD,
        OVERWRITE_OR_IGNORE TRUE
    )
""")


# ============================================================
# VALIDATION
# ============================================================

summary = con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT user) AS users

    FROM read_parquet(
        '{output}'
    )
""").df()


column_count = con.execute(f"""
    SELECT COUNT(*)
    FROM (
        DESCRIBE
        SELECT *
        FROM read_parquet(
            '{output}'
        )
    )
""").fetchone()[0]


duplicates = con.execute(f"""
    SELECT COUNT(*)

    FROM (
        SELECT
            user,
            day,
            COUNT(*) AS n

        FROM read_parquet(
            '{output}'
        )

        GROUP BY
            user,
            day

        HAVING COUNT(*) > 1
    )
""").fetchone()[0]


missing_baseline = con.execute(f"""
    SELECT COUNT(*)

    FROM read_parquet(
        '{output}'
    )

    WHERE baseline_workdays IS NULL
""").fetchone()[0]


known_case = con.execute(f"""
    SELECT
        user,
        day,
        files_copied_to_usb,
        usb_files_p95,

        ROUND(
            usb_files_p95_ratio,
            2
        ) AS usb_ratio,

        usb_files_above_p95

    FROM read_parquet(
        '{output}'
    )

    WHERE user = 'AAF0535'
      AND day = DATE '2010-08-10'
""").df()


# ============================================================
# ASSERTIONS
# ============================================================

rows = int(
    summary.iloc[0]["rows"]
)

users = int(
    summary.iloc[0]["users"]
)

assert rows == 464687, (
    f"Unexpected row count: {rows}"
)

assert users == 1000, (
    f"Unexpected user count: {users}"
)

assert column_count == 73, (
    f"Unexpected column count: {column_count}"
)

assert duplicates == 0, (
    f"Duplicate User-Day records: {duplicates}"
)

assert missing_baseline == 0, (
    f"Rows missing baseline: {missing_baseline}"
)

assert len(known_case) == 1

assert int(
    known_case.iloc[0][
        "files_copied_to_usb"
    ]
) == 11

assert float(
    known_case.iloc[0][
        "usb_files_p95"
    ]
) == 5.0

assert round(
    float(
        known_case.iloc[0][
            "usb_ratio"
        ]
    ),
    2,
) == 2.20

assert int(
    known_case.iloc[0][
        "usb_files_above_p95"
    ]
) == 1


# ============================================================
# FINAL STATUS
# ============================================================

print("\nBehavior-feature summary:")
display(summary)

print(
    f"\nTotal columns       : "
    f"{column_count}"
)

print(
    f"Duplicate User-Days : "
    f"{duplicates}"
)

print(
    f"Missing baselines   : "
    f"{missing_baseline}"
)

print("\nKnown CERT case:")
display(known_case)

print("\n" + "=" * 70)
print("✅ BUILD BEHAVIOR FEATURES: PASS")
print("All validation checks passed.")
print("=" * 70)

print(
    f"\nSaved to:\n"
    f"{BEHAVIOR_FEATURES_FILE}"
)

con.close()

✓ Required input files found.

BUILDING USER-DAY BEHAVIOR FEATURES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Behavior-feature summary:


,rows,users
0,464687,1000



Total columns       : 73
Duplicate User-Days : 0
Missing baselines   : 0

Known CERT case:


,user,day,files_copied_to_usb,usb_files_p95,usb_ratio,usb_files_above_p95
0,AAF0535,2010-08-10,11,5.0,2.2,1



✅ BUILD BEHAVIOR FEATURES: PASS
All validation checks passed.

Saved to:
C:\Users\HP\Documents\FYP\Data_engineering\data\features\user_day_behavior_features_v1.parquet


In [3]:
# ============================================================
# KNOWN CERT CASE: AAF0535 BEHAVIOR
# ============================================================

output = BEHAVIOR_FEATURES_FILE.resolve().as_posix()

con = duckdb.connect()

aaf_behavior = con.execute(f"""
    SELECT
        day,

        files_copied_to_usb,
        usb_files_p95,

        ROUND(
            usb_files_p95_ratio,
            2
        ) AS usb_ratio,

        usb_files_above_p95,

        external_email_events,
        external_email_p95,

        usb_files_7d,
        usb_files_14d,

        abnormal_day,
        abnormal_days_7d,

        activity_channel_diversity_7d

    FROM read_parquet(
        '{output}'
    )

    WHERE user = 'AAF0535'

      AND day BETWEEN
          DATE '2010-08-01'
          AND DATE '2010-08-20'

    ORDER BY day
""").df()

con.close()


print("=" * 70)
print("AAF0535 BEHAVIOR FEATURE INSPECTION")
print("=" * 70)

print(
    "\nObservation period: "
    "2010-08-01 → 2010-08-20"
)

display(aaf_behavior)

print(
    "\n✅ BEHAVIOR FEATURE INSPECTION COMPLETE"
)

AAF0535 BEHAVIOR FEATURE INSPECTION

Observation period: 2010-08-01 → 2010-08-20


,day,files_copied_to_usb,usb_files_p95,usb_ratio,usb_files_above_p95,external_email_events,external_email_p95,usb_files_7d,usb_files_14d,abnormal_day,abnormal_days_7d,activity_channel_diversity_7d
0,2010-08-01,0,5.0,0.0,0,0.0,3.0,10.0,18.0,0,1.0,3
1,2010-08-02,5,5.0,1.0,0,3.0,3.0,15.0,23.0,0,0.0,3
2,2010-08-03,6,5.0,1.2,1,2.0,3.0,19.0,29.0,1,1.0,3
3,2010-08-04,1,5.0,0.2,0,1.0,3.0,20.0,25.0,0,1.0,3
4,2010-08-05,3,5.0,0.6,0,0.0,3.0,19.0,26.0,1,2.0,3
5,2010-08-06,1,5.0,0.2,0,1.0,3.0,16.0,26.0,0,2.0,3
6,2010-08-07,0,5.0,0.0,0,0.0,3.0,16.0,26.0,0,2.0,3
7,2010-08-08,0,5.0,0.0,0,0.0,3.0,16.0,26.0,0,2.0,3
8,2010-08-09,5,5.0,1.0,0,2.0,3.0,16.0,31.0,1,3.0,3
9,2010-08-10,11,5.0,2.2,1,1.0,3.0,21.0,40.0,1,3.0,3



✅ BEHAVIOR FEATURE INSPECTION COMPLETE


In [5]:
# ============================================================
# RAW EMAIL CLASSIFICATION VERIFICATION
# ============================================================

email_path = EMAIL_CSV.resolve().as_posix()

con = duckdb.connect()

result = con.execute(f"""
    WITH email_recipients AS (

        SELECT
            lower(
                concat_ws(
                    ';',
                    coalesce("to", ''),
                    coalesce(cc, ''),
                    coalesce(bcc, '')
                )
            ) AS recipients

        FROM read_csv(
            '{email_path}',
            header = true,
            all_varchar = true
        )
    ),

    classified AS (

        SELECT
            recipients,

            regexp_replace(
                recipients,
                '[^;]*@dtaa[.]com',
                '',
                'g'
            ) AS after_internal_removed

        FROM email_recipients
    )

    SELECT
        COUNT(*) AS total_emails,

        SUM(
            CASE
                WHEN position(
                    '@'
                    IN after_internal_removed
                ) > 0
                THEN 1
                ELSE 0
            END
        ) AS external_email_events,

        ROUND(
            100.0
            *
            SUM(
                CASE
                    WHEN position(
                        '@'
                        IN after_internal_removed
                    ) > 0
                    THEN 1
                    ELSE 0
                END
            )
            / COUNT(*),
            2
        ) AS external_percentage

    FROM classified
""").df()

con.close()


print("=" * 70)
print("EXTERNAL EMAIL CLASSIFICATION CHECK")
print("=" * 70)

display(result)


total_emails = int(
    result.iloc[0][
        "total_emails"
    ]
)

external_emails = int(
    result.iloc[0][
        "external_email_events"
    ]
)

external_percentage = float(
    result.iloc[0][
        "external_percentage"
    ]
)


assert total_emails == 2629979

assert external_emails == 1264049

assert abs(
    external_percentage - 48.06
) < 0.01


print(
    "\n✅ EXTERNAL EMAIL "
    "CLASSIFICATION VERIFIED"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

EXTERNAL EMAIL CLASSIFICATION CHECK


,total_emails,external_email_events,external_percentage
0,2629979,1264049.0,48.06



✅ EXTERNAL EMAIL CLASSIFICATION VERIFIED


In [7]:
# ============================================================
# VERIFY EMBEDDED USER BASELINE
# ============================================================

behavior = (
    BEHAVIOR_FEATURES_FILE
    .resolve()
    .as_posix()
)

con = duckdb.connect()

baseline_check = con.execute(f"""
    SELECT DISTINCT
        user,
        baseline_workdays,

        external_email_median,
        external_email_p95,

        usb_files_median,
        usb_files_p95,

        http_requests_median,
        http_requests_p95

    FROM read_parquet(
        '{behavior}'
    )

    WHERE user = 'AAF0535'
""").df()

con.close()


print("=" * 70)
print("EMBEDDED BASELINE CHECK: AAF0535")
print("=" * 70)

display(baseline_check)


assert len(
    baseline_check
) == 1

assert int(
    baseline_check.iloc[0][
        "baseline_workdays"
    ]
) == 111

assert float(
    baseline_check.iloc[0][
        "external_email_p95"
    ]
) == 3.0

assert float(
    baseline_check.iloc[0][
        "usb_files_p95"
    ]
) == 5.0

assert float(
    baseline_check.iloc[0][
        "http_requests_p95"
    ]
) == 29.0


print(
    "\n✅ EMBEDDED BASELINE VALUES VERIFIED"
)

EMBEDDED BASELINE CHECK: AAF0535


,user,baseline_workdays,external_email_median,external_email_p95,usb_files_median,usb_files_p95,http_requests_median,http_requests_p95
0,AAF0535,111,1.0,3.0,2.0,5.0,29.0,29.0



✅ EMBEDDED BASELINE VALUES VERIFIED
